Here’s a concise theory section you could present to users before they start exploring your simulator. It sets the stage without overwhelming them, and highlights exactly what they need to know to make sense of the plots and controls.

---

## 📖 Brief Theory for Using the Diode Simulation

### 1. The p–n Junction
- A diode is formed by joining **p-type** (acceptor doped) and **n-type** (donor doped) semiconductors.  
- At the junction, electrons and holes diffuse and leave behind fixed ions, creating a **depletion region**.  
- This region acts like a barrier, with a built-in potential \(V_{bi}\).

### 2. Depletion Region and Barrier
- The depletion width depends on doping levels \(N_A\) and \(N_D\).  
- Heavier doping → narrower depletion region, lower breakdown voltage.  
- Lighter doping → wider depletion region, higher breakdown voltage.  
- The simulator shows this barrier ramp in **atomic spacings** for intuition.

### 3. Forward Bias
- Applying a positive voltage reduces the barrier.  
- Once the applied bias exceeds \(V_{bi}\), carriers flow freely.  
- Current rises **exponentially** with voltage, making the diode an excellent conductor in forward bias.

### 4. Reverse Bias
- Applying a negative voltage increases the barrier.  
- Current is ideally just the tiny **saturation current** \(I_s\), nearly flat at picoamp levels.  
- In real diodes, leakage grows at high voltages, but within ±2 V it remains negligible (except for Zener diodes).  
- This is why diodes act as **good switches**: they block current until breakdown.

### 5. Series Resistance
- A practical diode has resistance \(R_s\) in series.  
- This limits current at high forward bias and shifts the I–V curve.  
- The simulator lets you vary \(R_s\) to see its effect.

### 6. Practical Diodes
- Different diodes (1N4148, 1N4007, BAT54, BZX55) have different doping levels and behaviors.  
- The dropdown lets you select a diode and see how its junction width and I–V curve differ.

---

## 🎯 How to Use the Simulator
1. **Select a diode** from the dropdown.  
2. **Adjust bias voltage** (slider in mV) to see forward and reverse behavior.  
3. **Change series resistance** to observe its effect on current.  
4. **Interpret plots**:
   - Top plot: depletion region barrier across the junction.  
   - Bottom left: forward bias I–V curve.  
   - Bottom right: reverse bias leakage (pA/µA scale).  
5. **Numeric readout** gives built-in potential, depletion width, and current values.

---

This brief theory ensures users understand the **physics behind the plots** and why the simulator behaves the way it does.  

Would you like me to format this into a **short “User Guide” section** that could be displayed alongside the simulator (like a collapsible info box), so users can quickly reference it while experimenting?

In [23]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown, Output
from scipy.optimize import fsolve

# Constants
q = 1.6e-19
k = 1.38e-23
T = 300
epsilon = 11.7 * 8.85e-14   # F/cm
n_i = 1.5e10                # cm^-3
a = 5.43e-8                 # cm
V_T = (k*T)/q

out = Output()

# Practical diode options
diode_options = {
    "1N4148 (small-signal Si)": (1e16, 1e16),
    "1N4007 (power rectifier)": (1e15, 1e15),
    "BAT54 (Schottky)": (1e17, 1e17),
    "BZX55 (Zener 5.1V)": (1e18, 1e18)
}

def diode_dual_plot(diode_type, V_applied_mV, R_s):
    N_A, N_D = diode_options[diode_type]
    V_applied = V_applied_mV / 1000.0
    
    V_bi = V_T * np.log((N_A * N_D) / (n_i**2))
    V_junction = max(V_bi - V_applied, 0)
    
    W = np.sqrt((2 * epsilon / q) * ((N_A + N_D) / (N_A * N_D)) * V_junction)
    N_atoms = int(W / a)
    max_atoms = N_atoms + 100
    
    x_atoms = np.arange(0, max_atoms)
    profile = np.zeros_like(x_atoms, dtype=float)
    
    left_boundary = 150
    right_boundary = left_boundary + N_atoms
    junction_center = (left_boundary + right_boundary) // 2
    
    mask = (x_atoms >= left_boundary) & (x_atoms <= right_boundary)
    if mask.sum() > 0:
        profile[mask] = np.linspace(0, V_junction*1000, mask.sum())
    
    # Layout: 2 rows (junction full width, then forward/reverse side by side)
    fig = plt.figure(figsize=(16,10), constrained_layout=True)
    gs = fig.add_gridspec(2, 2)
    
    # --- Row 1: Junction view (full width) ---
    ax1 = fig.add_subplot(gs[0, :])
    ax1.axvspan(0, left_boundary, color="plum", alpha=0.3, label="p-region")
    ax1.axvspan(left_boundary, junction_center, color="orange", alpha=0.3, label="fixed negative ions")
    ax1.axvspan(junction_center, right_boundary, color="orange", alpha=0.3, label="fixed positive ions")
    ax1.axvspan(right_boundary, max_atoms, color="cyan", alpha=0.3, label="n-region")
    
    ax1.plot(x_atoms, profile, 'b-', linewidth=2, label="Effective barrier ramp")
    ax1.axhline(V_bi*1000, color="red", linestyle="--", linewidth=2, label="Built-in potential")
    ax1.axvline(junction_center, color="black", linestyle="--", label="Junction center")
    
    ax1.set_xlabel("Atomic distance count")
    ax1.set_ylabel("Barrier height (mV)")
    ax1.set_ylim(0, max(800, V_junction*1000*1.2))
    ax1.set_title(f"{diode_type}\nDepletion region width ≈ {N_atoms} atoms")
    ax1.legend(loc="upper right")
    ax1.grid(True)
    
    # --- Row 2: Forward bias (left) and Reverse bias (right) ---
    I_s = 1e-12
    def diode_current(V):
        func = lambda I: I - I_s*(np.exp((V - I*R_s)/V_T) - 1)
        return fsolve(func, 0)[0]
    
    V = np.linspace(-2, 2, 400)
    I = np.array([diode_current(v) for v in V])
    I_op = diode_current(V_applied)
    
    # Forward bias plot
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.plot(V, I, 'g-', label=f"I-V curve (R_s={R_s:.1f} Ω)")
    ax2.plot(V_applied, I_op, 'ro', label="Operating point")
    ax2.set_xlim(0, 2)
    ax2.set_ylim(0, 1e-2)
    ax2.set_xlabel("Forward bias (V)")
    ax2.set_ylabel("Current (A)")
    ax2.set_title("Forward bias characteristic")
    ax2.legend(loc="upper left")
    ax2.grid(True)
    
    # Reverse bias plot
    ax3 = fig.add_subplot(gs[1, 1])
    if np.max(np.abs(I)) >= 1e-6: # Use microamp scale 
        ax3.plot(V, I*1e6, 'r--', label="Leakage current (µA)") 
        ax3.plot(V_applied, I_op*1e6, 'ro', alpha=0.6) 
        ax3.set_ylabel("Leakage current (µA)") 
        ax3.set_ylim(-10, 10) # adjust as needed 
        leakage_val = I_op*1e6 
        leakage_unit = "µA" 
    else: # Use picoamp scale 
        ax3.plot(V, I*1e12, 'r--', label="Leakage current (pA)") 
        ax3.plot(V_applied, I_op*1e12, 'ro', alpha=0.6) 
        ax3.set_ylabel("Leakage current (pA)") 
        ax3.set_ylim(-1000, 100) # adjust as needed 
        leakage_val = I_op*1e12 
        leakage_unit = "pA"    
    ax3.set_xlim(-2, 0)
    ax3.set_ylim(-1000, 100)
    ax3.set_xlabel("Reverse bias (V)")    
    ax3.set_title("Reverse bias leakage")
    ax3.legend(loc="upper left")
    ax3.grid(True)
    
    plt.show()
    
    # Numeric readout
    with out:
        out.clear_output()
        print(f"Selected diode: {diode_type}")
        print(f"Applied bias: {V_applied:.3f} V")
        print(f"Built-in potential: {V_bi:.3f} V")
        print(f"Effective barrier: {V_junction:.3f} V")
        print(f"Depletion width: {N_atoms} atoms (~{W*1e7:.2f} nm)")
        print(f"Diode current with R_s={R_s:.1f} Ω: {I_op:.3e} A")
        print(f"Leakage current: {leakage_val:.3f} {leakage_unit}")        

# Controls
bias_slider = FloatSlider(value=0, min=-2000, max=2000, step=10,
                          description="Applied bias (mV)", layout={'width': '800px'})
interact(
    diode_dual_plot,
    diode_type=Dropdown(options=list(diode_options.keys()), description="Select Diode"),
    V_applied_mV=bias_slider,
    R_s=FloatSlider(value=10, min=0, max=100, step=1, description="Series R (Ω)")
)

display(out)

interactive(children=(Dropdown(description='Select Diode', options=('1N4148 (small-signal Si)', '1N4007 (power…

Output()